In [ ]:
%pip install opencv-python

In [17]:
import cv2
import numpy as np

In [26]:
# Global kontrol değişkenleri
captured_frame = None
show_results = False

def mouse_event_handler(event, x, y, flags, param):
    global captured_frame, show_results
    if event == cv2.EVENT_LBUTTONDOWN:
        if not show_results:
            captured_frame = param.copy()
            show_results = True
            print(">> Görüntü yakalandı!")
    elif event == cv2.EVENT_LBUTTONDBLCLK:
        show_results = False
        captured_frame = None
        try:
            cv2.destroyWindow("4 Farkli Perspektif Paneli")
        except:
            pass
        print(">> Sistem sıfırlandı.")

def get_perspective(img, src_pts, dst_pts, label, size=(300, 300)):
    # Matris hesaplama ve dönüştürme
    matrix = cv2.getPerspectiveTransform(src_pts, dst_pts)
    warped = cv2.warpPerspective(img, matrix, size)
    
    # Metni sol alt köşeye ekle (Y koordinatı: size[1] - 10)
    # cv2.putText(görüntü, metin, (x, y), font, ölçek, renk, kalınlık)
    cv2.putText(warped, label, (10, size[1] - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    return warped

cap = cv2.VideoCapture(0)
cv2.namedWindow("Ana Kamera - Hizalama")

while True:
    ret, frame = cap.read()
    if not ret: break
    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    margin = 150
    pts1 = np.float32([[margin, margin], [w-margin, margin], 
                       [w-margin, h-margin], [margin, h-margin]])

    cv2.setMouseCallback("Ana Kamera - Hizalama", mouse_event_handler, frame)

    display_frame = frame.copy()
    cv2.polylines(display_frame, [np.int32(pts1)], True, (0, 255, 0), 3)
    
    status_text = "Sifirlamak icin CIFT TIKLA" if show_results else "Hizala ve TEK TIKLA"
    cv2.putText(display_frame, status_text, (20, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.imshow("Ana Kamera - Hizalama", display_frame)

    if show_results and captured_frame is not None:
        s = 300 
        
        # 4 Farklı Perspektif Tanımı
        # 1. Kuş Bakışı: Kaynak noktaları tam kareye yayar
        dst_std = np.float32([[0, 0], [s, 0], [s, s], [0, s]])
        res1 = get_perspective(captured_frame, pts1, dst_std, "1. Kus Bakisi (Duz)")
        
        # 2. Sağdan Bakış: Sağ kenarı daraltarak derinlik verir
        dst_right = np.float32([[0, 0], [s, 70], [s, s-70], [0, s]])
        res2 = get_perspective(captured_frame, pts1, dst_right, "2. Sagdan Bakis")
        
        # 3. Soldan Bakış: Sol kenarı daraltarak derinlik verir
        dst_left = np.float32([[0, 70], [s, 0], [s, s], [0, s-70]])
        res3 = get_perspective(captured_frame, pts1, dst_left, "3. Soldan Bakis")
        
        # 4. Yatay Eğim: Üst tarafı daraltıp altı geniş tutar
        dst_tilt = np.float32([[70, 0], [s-70, 0], [s, s], [0, s]])
        res4 = get_perspective(captured_frame, pts1, dst_tilt, "4. Yatay Egim")

        # Paneli birleştir ve göster
        panel = np.vstack((np.hstack((res1, res2)), np.hstack((res3, res4))))
        cv2.imshow("4 Farkli Perspektif Paneli", panel)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()